### Assemble context for a RAG answer

Your retriever returns text chunks ranked from most to least relevant. Before sending them to the LLM, you need to select which chunks fit within a token budget. Each chunk is a (chunk_id, token_count) pair.

def select_context(chunks: list[tuple[str, int]], budget: int) -> list[str]:
    ...

Read the chunks in the supplied order. Select a chunk if its ID hasn’t already been selected and its token count fits the remaining budget. Each selected ID appears once in the output, in selection order. If a chunk is too large, skip it and keep looking. Assume token counts are positive, the budget is nonnegative, and repeated IDs have the same token count.


chunks = [
    ("policy-A", 6),
    ("guide-B", 5),
    ("policy-A", 6),
    ("faq-C", 4),
]
budget = 10

First question: Working through this example by hand, which chunk IDs would you return?


In [16]:
def select_context(chunks: list[tuple[str, int]], budget: int) -> list[str]:
    seen = set()
    # this preserves the order
    selected = []
    for chunk, size in chunks:     
        if size > budget:
            continue   
        elif size <= budget and chunk not in seen:
            seen.add((chunk))
            selected.append(chunk)
            budget -= size
    return (selected)

# usage 
result = select_context([("A", 6), ("B", 4)], budget=4)

print("Actual result:", result)
assert result == ["B"], f"Expected ['B'], got {result!r}"
print("Test passed")


Actual result: ['B']
Test passed


In [11]:
def select_context(chunks: list[tuple[str, int]], budget: int) -> list[str]:
    selection = set()
    for chunk, size in chunks:        
        while size <= budget and chunk not in selection:
            selection.add((chunk))
            budget -= size
    return (list(selection))

# Usage 
chunks = [
    ("policy-A", 6),
    ("guide-B", 5),
    ("policy-A", 6),
    ("faq-C", 4),
]
budget = 10
print (select_context(chunks, budget))


['policy-A', 'faq-C']
